# 00 - Coverage (Q1, run first)

Do sector labels exist in volume? Label counts and sector balance.

**Pass (pre-registered in `config.yaml` → `gates.q1`):** >= 2,000 single-sector CVEs,
>= 5 sectors with >= 50 CVEs each, no sector above 60% of labeled CVEs. Otherwise pivot to time-to-KEV.

In [1]:
import pandas as pd
from src.config import load_config, data_path

pd.set_option("display.width", 140)
cfg = load_config()
interim, processed = data_path(cfg, "interim"), data_path(cfg, "processed")

In [2]:
joined = pd.read_parquet(interim / "joined.parquet")
labels = pd.read_parquet(interim / "labels_sector.parquet")
funnel = pd.Series({
    "NVD CVEs published in window": len(joined),
    "listed in a CISA ICS advisory": int(joined["in_ics"].sum()),
    ">= 1 canonical sector": int((labels["n_sectors"] >= 1).sum()),
    "exactly 1 sector (primary task)": int(labels["sector"].notna().sum()),
    "2+ sectors (multi-label only)": int((labels["n_sectors"] >= 2).sum()),
    "'Multiple Sectors' / unparsable": int((labels["n_sectors"] == 0).sum()),
    "in KEV (time-to-KEV fallback)": int(joined["in_kev"].sum()),
})
funnel.to_frame("CVEs")

,CVEs
NVD CVEs published in window,113475
listed in a CISA ICS advisory,4986
>= 1 canonical sector,4125
exactly 1 sector (primary task),2037
2+ sectors (multi-label only),2088
'Multiple Sectors' / unparsable,861
in KEV (time-to-KEV fallback),522


## Primary task: single-sector CVEs

In [3]:
single = labels.dropna(subset=["sector"])
dist = single["sector"].value_counts().to_frame("CVEs")
dist["share"] = (dist["CVEs"] / dist["CVEs"].sum()).round(3)
dist["vendors"] = single.groupby("sector")["vendor"].nunique()
dist

,CVEs,share,vendors
sector,,,
Critical Manufacturing,1262,0.620,81
Energy,408,0.200,30
Healthcare and Public Health,110,0.054,28
Communications,93,0.046,29
Commercial Facilities,83,0.041,41
Information Technology,34,0.017,11
Transportation Systems,28,0.014,14
Emergency Services,9,0.004,3
Water and Wastewater Systems,7,0.003,3


## Multi-label prevalence (any CVE with >= 1 sector)

In [4]:
multi = labels[labels["n_sectors"] >= 1]
prev = multi.explode("sectors")["sectors"].value_counts().to_frame("CVEs")
prev["prevalence"] = (prev["CVEs"] / len(multi)).round(3)
prev

,CVEs,prevalence
sectors,,
Critical Manufacturing,3142,0.762
Energy,2337,0.567
Transportation Systems,1589,0.385
Healthcare and Public Health,1448,0.351
Government Facilities,1381,0.335
Financial Services,1294,0.314
Commercial Facilities,353,0.086
Water and Wastewater Systems,352,0.085
Communications,195,0.047


## Vendor concentration

A sector classifier trained on a handful of vendors learns vendors, not sectors.
This is why Q2 is also scored on a vendor-grouped split.

In [5]:
top = single["vendor"].value_counts().head(10).to_frame("CVEs")
top["share"] = (top["CVEs"] / len(single)).round(3)
top["top sector"] = single.groupby("vendor")["sector"].agg(lambda s: s.value_counts().index[0])
print(f"{single['vendor'].nunique()} vendors across {len(single)} single-sector CVEs")
top

224 vendors across 2037 single-sector CVEs


,CVEs,share,top sector
vendor,,,
siemens,757,0.372,Critical Manufacturing
rockwell_automation,166,0.081,Critical Manufacturing
abb,78,0.038,Energy
delta_electronics,77,0.038,Critical Manufacturing
mitsubishi_electric,75,0.037,Critical Manufacturing
hitachi_energy,69,0.034,Energy
schneider_electric,33,0.016,Critical Manufacturing
growatt,30,0.015,Energy
fuji_electric,20,0.010,Critical Manufacturing


## Fallback target: time-to-KEV

In [6]:
ttk = pd.read_parquet(interim / "labels_time_to_kev.parquet")
print(ttk["days_to_kev"].describe().round(1).to_dict())
ttk["time_to_kev"].value_counts().to_frame("CVEs")

{'count': 522.0, 'mean': 90.8, 'std': 188.3, 'min': -56.0, '25%': 0.0, '50%': 7.5, '75%': 89.8, 'max': 1268.0}


,CVEs
time_to_kev,
<=7d,261
8-90d,131
>90d,130


## Q1 decision

In [7]:
from src.gates import q1_coverage

q1 = q1_coverage(labels, cfg)
q1

{'labeled': 2037,
 'sectors_at_min_size': 5,
 'max_share': 0.6195385370643103,
 'max_sector': 'Critical Manufacturing',
 'checks': {'min_labeled': True, 'min_sectors': True, 'max_share': False},
 'pass': False}